In [17]:
%matplotlib inline

%reload_ext autoreload
%autoreload 2
from imports import *


In [18]:
from src.utils import pmf_utils, classification_utils
from config import dir_config, main_config

## Load config and data

In [19]:
processed_dir = Path(dir_config.data.processed)

psych_model_type = main_config.BEHAVIOR.psych_model

In [20]:
aggregate_data = pd.read_csv(Path(processed_dir, "processed_all_data_accu_60_all.csv"), index_col=None)
valid_data = pd.read_csv(Path(processed_dir, "processed_all_data_accu_60_filtered.csv"), index_col=None)
processed_metadata = pd.read_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), encoding="latin1", index_col=None)

In [21]:
subjects_map, grp_indices = classification_utils.get_subject_classification_ids(processed_metadata)

PD subjects with ON+OFF sessions: 41
PD subjects with UPDRS subtype: 22
All PD subjects:           41
  Tremor dominant:         11 	 ['P3' 'P6' 'P7' 'P11' 'P12' 'P17' 'P18' 'P19' 'P29' 'P31' 'P32']
  Brady dominant:          10 	 ['P1' 'P4' 'P9' 'P13' 'P20' 'P22' 'P23' 'P28' 'P33' 'P34']
  Intermediate:            1 	 ['P24']
HC subjects:               18 	 ['HC1' 'HC3' 'HC6' 'HC7' 'HC8' 'HC9' 'HC12' 'HC13' 'AV' 'BC' 'BF' 'EM'
 'ES' 'GF' 'GP' 'JA' 'MRM' 'SY']


### Adding stimulus-specific PMF params

In [22]:
psych_vars = ["bias", "psych_bias", "psych_alpha", "psych_beta", "psych_lapse", "psych_guess"]
for var in psych_vars:
    processed_metadata[f"overall_{var}"] = np.nan
    processed_metadata[f"equal_{var}"] = np.nan
    processed_metadata[f"positive_{var}"] = np.nan

In [23]:
session_groups = [
    ("off", grp_indices["pd_off"]),
    ("on", grp_indices["pd_on"]),
    (None, grp_indices["healthy"]),
]

for med, group_idx in session_groups:
    for current_idx in group_idx:
        subject = processed_metadata.loc[current_idx, "subject_id"]
        if med:
            session_data = valid_data[(valid_data["subject_id"] == subject) & (valid_data["medication"] == med)]
        else:
            session_data = valid_data[valid_data["subject_id"] == subject]

        coh, overall_psych, overall_model, _, _ = pmf_utils.get_psychometric_data(data=session_data, model_type=psych_model_type)
        coh, positive_psych, positive_model, _, _ = pmf_utils.get_psychometric_data(data=session_data[session_data["color"] == 1], model_type=psych_model_type)
        coh, equal_psych, equal_model, _, _ = pmf_utils.get_psychometric_data(data=session_data[session_data["color"] == 0], model_type=psych_model_type)

        processed_metadata.loc[current_idx, "overall_bias"] = overall_psych[3]
        processed_metadata.loc[current_idx, "overall_psych_bias"] = overall_model.predict(0)
        processed_metadata.loc[current_idx, "overall_psych_alpha"] = overall_model.coefs_["mean"]
        processed_metadata.loc[current_idx, "overall_psych_beta"] = overall_model.coefs_["var"]
        processed_metadata.loc[current_idx, "overall_psych_lapse"] = overall_model.coefs_["lapse_rate"]
        processed_metadata.loc[current_idx, "overall_psych_guess"] = overall_model.coefs_["guess_rate"]

        processed_metadata.loc[current_idx, "positive_bias"] = positive_psych[3]
        processed_metadata.loc[current_idx, "positive_psych_bias"] = positive_model.predict(0)
        processed_metadata.loc[current_idx, "positive_psych_alpha"] = positive_model.coefs_["mean"]
        processed_metadata.loc[current_idx, "positive_psych_beta"] = positive_model.coefs_["var"]
        processed_metadata.loc[current_idx, "positive_psych_lapse"] = positive_model.coefs_["lapse_rate"]
        processed_metadata.loc[current_idx, "positive_psych_guess"] = positive_model.coefs_["guess_rate"]

        processed_metadata.loc[current_idx, "equal_bias"] = equal_psych[3]
        processed_metadata.loc[current_idx, "equal_psych_bias"] = equal_model.predict(0)
        processed_metadata.loc[current_idx, "equal_psych_alpha"] = equal_model.coefs_["mean"]
        processed_metadata.loc[current_idx, "equal_psych_beta"] = equal_model.coefs_["var"]
        processed_metadata.loc[current_idx, "equal_psych_lapse"] = equal_model.coefs_["lapse_rate"]
        processed_metadata.loc[current_idx, "equal_psych_guess"] = equal_model.coefs_["guess_rate"]


In [24]:
processed_metadata.to_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), index=False)